In [ ]:
import pandas as pd
from pathlib import Path
from functools import reduce

from parreg.process_config import load_and_validate_config
from parreg import utils

In [6]:
config_file = Path('/home/yuqiong.liu/work/Gitlab/ngen-regionalization/configs/config.yaml')
if not config_file.exists():
    raise FileNotFoundError(config_file)

config = load_and_validate_config(config_file)

datasets = config.general.attr_dataset_list
out1 = config.file_io.output['attr_data_final']
out2 = config.file_io.output['config_final']

for vpu in config.general.vpu_list:
    df_attrs_all = []
    for dataset_name in datasets:

        print(f'vpu={vpu}, dataset={dataset_name}')
        dataset = config.file_io.attr_datasets[dataset_name]
        df_attrs = dataset.get_attr_data()
        df_attrs = df_attrs.rename(columns=lambda x: x if x == "divide_id" else f"{dataset_name}_{x}")
        df_attrs_all.append(df_attrs)

    # Merge all attribute data frames column-wise, based on divide_id
    df_attrs_all= reduce(
        lambda left, right: pd.merge(left, right, on='divide_id', how='outer'),
        df_attrs_all
    )

    # check percentage of missing data
    #print(df_attrs_all.isna().mean()*100)

    if out1.save:
        if not Path(out1.path).is_dir():
            raise ValueError(f'')
        utils.save_data(df_attrs_all, Path(out1.path, 'attr_' + config.general.domain + '_vpu' + vpu + '.' + out1.format))

if out2.save:
    if Path(out2.path).is_file():
        utils.save_data(config, Path(out2.path))
    elif Path(out2.path).is_dir():                
        utils.save_data(config, Path(out2.path, 'config_final.' + out2.format))
    

vpu=01, dataset=ngen
vpu=01, dataset=hlr
vpu=02, dataset=ngen
vpu=02, dataset=hlr
